<a href="https://colab.research.google.com/github/Marcin19721205/WSBNeuronowe/blob/main/CW4_Rownowazenie_Klas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wprowadzenie

Kolejnym ważnym aspektem niemal każdego eksperymentu uczenia maszynowego jest możliwość równoważenia klas w przypadku, gdy ich proporcje są zaburzone. W większości rzeczywistych zadań, część klas może występować w niewielkiej ilości. Niestety, najczęściej będą to te najbardziej wartościowe i istotne. Istnieją narzędzia, które pozwalają radzić sobie z takimi zjawiskami.

W tym notebooku zapoznamy się z podstawowymi technikami, które mogą nam pomóc przywrócić właściwe proporcje w danych.

In [1]:
%pip install imblearn

In [16]:
%pip install tensorflow-addons

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
#import tensorflow_addons as tfa
import gc
import tensorflow.keras as krs

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, accuracy_score, precision_score, recall_score

In [20]:
%matplotlib inline

# Wczytanie danych



W ramach tego ćwiczenia będziemy pracować na zbiorze danych do klasyfikacji wieloklasowej, gdzie porporcje pomiędzy klasami są dość silnie zaburzone.

<div class='alert alert-block alert-warning'>
    Wczytaj zbiór danych o nazwie <b>imbalanced_dataset.csv</b>. Sprawdź proporcje pomiędzy klasami, zawartymi w kolumnie <b>y</b>.
</div>

Dane są dostępne pod adresem `https://drive.google.com/uc?id=1u2M_HRvD_MXJ7Kztbyr7i9FvvW0Ms6TC&export=download`

In [21]:
data = pd.read_csv("https://drive.google.com/uc?id=1u2M_HRvD_MXJ7Kztbyr7i9FvvW0Ms6TC&export=download")

In [22]:
data['y'].value_counts(normalize=True)

,proportion
y,
0,0.6980
1,0.2012
2,0.1008


<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>:
    <ol>
        <li>oddziel kolumnę y od całej reszty danych. Zapisz ją pod zmienną y</li>
        <li>dane, które pozostają - zapisz pod zmienną X</li>
        <li>zrób rzutowanie typów zmiennej X na typ <code>np.float32</code> ze względu na kompatybilność z tensorflow</li>
        <li>zamień klasy wektora <code>y</code> na postać one-hot i zapisz ponownie pod zmienną y</li>
    </ol>
</div>

In [23]:
y = data.pop('y') # oddziel kolumnę y od całej reszty danych
X = data # dane, które pozostają - zapisz pod zmienną X
X = X.astype(np.float32) # rzutowanie typów zmiennej X na typ np.float32

In [24]:
y = tf.keras.utils.to_categorical(y, num_classes=3) # zamień klasy wektora y na postać one-hot

Sprawdzenie poprawności wyników:

In [25]:
assert X.shape == (5000, 20)
assert (X.dtypes == np.float32).all()

assert y.shape == (5000, 3)

<div class='alert alert-block alert-warning'>
    Podziel dane na train i test w proporcji <code>train = 0.8% zbioru, random_state = 123</code>
</div>

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123) # podziel zbiór

Sprawdzenie poprawności wyniku

In [27]:
assert X_train.shape == (4000, 20)
assert y_train.shape == (4000, 3)

assert X_test.shape == (1000, 20)
assert y_test.shape == (1000, 3)

assert X_train.values.dtype == np.float32
assert X_test.values.dtype == np.float32

# Bazowe modele

Zaczniemy od zbudowania bazowego modelu, który będzie operował na oryginalnych danych, bez równoważenia. Stwrzomy dwa modele:

1. Zawsze przewidujący najczęstszą klasę
2. Prostą sieć neuronową do klasyfikacji wieloklasowej

Sieć neuronową o zadanej architekturze będziemy szkolić od zera kilkukrotnie, odpowiednio manipulując wcześniej danymi.

<div class='alert alert-block alert-danger'>
    <b>UWAGA</b> w tym ćwiczeniu nie skupiamy się na stworzeniu jak najlepszej architektury sieci neurnowej dla zadanego problemu. Chcemy za to zbadać wpływ równowagi klas lub jej braku na jakość predykcji. Nie skupiaj się więc na aspekcie doboru jak najlepszej sieci, ale na operacjach na danych, które za chwilę będziemy wykonywać.
</div>

## Prosty klasyfikator

<div class='alert alert-block alert-warning'>
    Zbuduj klasyfikator, zawsze przewidujący najczęściej występującą klasę. Wykorzystaj implementajcę <code>sklearn.dummy.DummyClassifier</code>
</div>


In [28]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent', random_state=123)
dummy.fit(X_train, y_train)

DummyClassifier(random_state=123, strategy='most_frequent')

Sprawdzenie poprawności wyniku

In [29]:
yhat_dummy = dummy.predict(X_test)
assert np.round(accuracy_score(y_test, yhat_dummy), 3) == 0.694
print(classification_report(y_test, yhat_dummy, zero_division=0))

clf_rep = classification_report(y_test, yhat_dummy, zero_division=0, output_dict=True)
assert np.round(clf_rep['0']['precision'], 3) == 0.694
assert clf_rep['1']['precision'] == 0.0

assert clf_rep['0']['recall'] == 1.0
assert clf_rep['1']['recall'] == 0.0

              precision    recall  f1-score   support

           0       0.69      1.00      0.82       694
           1       0.00      0.00      0.00       215
           2       0.00      0.00      0.00        91

   micro avg       0.69      0.69      0.69      1000
   macro avg       0.23      0.33      0.27      1000
weighted avg       0.48      0.69      0.57      1000
 samples avg       0.69      0.69      0.69      1000



<div class='alert alert-block alert-info'>
    W powyższych wynikach widać trzy niepokojące rzeczy:
    
<ol>
<li>Głupi klasyfikator potrafi "osiągnąć" trafnośc na poziomie 69%</li>
<li>Metryki takie jak trafność są bezużyteczne w przypadku braku zrównoważenia klas</li>
<li>Dopiero łączne wykorzystanie metryk precyzji, czułości oraz F1 pozwala zobaczyć skalę problemu</li>
</ol>
</div>

## Sieć neuronowa

<div class='alert alert-block alert-warning'>
    Przygotuj funkję, która buduje i zwraca gotową sieć neuronową o następującej specyfikacji:

<ol>
<li>Warstwy: <code>BatchNorm - Dense(32, relu) - Dense(16, relu) - Dense(3, softmax)</code></li>
<li>Dodatkowe opcje: <code>optymalizator=Adam, koszt=categorical_crossentropy, metryki: accuracy, F1Score(num_classes=3, average=macro)</code></li>
</ol>
</div>

<div class='alert alert-block alert-info'>
    Metryka F1Score zawarta są w bardzo przydatnym pakiecie <code>tensorflow_addons</code>. Warto przeczytać dokumentację tego narzędzia.
</div>


In [33]:
def build_model():
    model = krs.Sequential([
        krs.layers.BatchNormalization(input_shape=(20,)),
        krs.layers.Dense(32, activation='relu'),
        krs.layers.Dense(16, activation='relu'),
        krs.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer='Adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [35]:
model1 = build_model()

Sprawdzenie poprawności wyniku:

In [36]:
assert 'batch_normalization' in model1.layers[0].name

assert 'dense' in model1.layers[1].name
assert model1.layers[1].units == 32
assert model1.layers[1].activation is tf.keras.activations.relu

assert 'dense' in model1.layers[2].name
assert model1.layers[2].units == 16
assert model1.layers[2].activation is tf.keras.activations.relu

assert 'dense' in model1.layers[3].name
assert model1.layers[3].units == 3
assert model1.layers[3].activation is tf.keras.activations.softmax

<div class='alert alert-block alert-warning'>
    Wyszkol przygotowany model przez 5 epok (batch size 32) na zbiorze treningowym. Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m1_eval.
</div>

In [37]:
model1.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)
m1_eval = model1.evaluate(X_test, y_test, return_dict=True)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6621 - loss: 0.8574
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7282 - loss: 0.7177
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7465 - loss: 0.6466
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7819 - loss: 0.5887
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7873 - loss: 0.5464
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8004 - loss: 0.5308


In [38]:
m1_eval

{'accuracy': 0.8040000200271606, 'loss': 0.5257893800735474}

Sprawdzenie poprawności wyniku:

In [39]:
assert m1_eval['accuracy'] >= 0.7

<div class='alert alert-block alert-warning'>
    Dokonaj predykcji na zbiorze testowym. Wszystkie obiekty, które osiągną próg pewności >=0.5 zalicz do klasy 1. Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.
</div>

In [40]:
y_pred = model1.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)
print(classification_report(y_test_classes, y_pred_classes))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

           0       0.82      0.98      0.89       694
           1       0.77      0.52      0.62       215
           2       0.46      0.14      0.22        91

    accuracy                           0.80      1000
   macro avg       0.69      0.55      0.58      1000
weighted avg       0.78      0.80      0.77      1000



<div class='alert alert-block alert-info'>
    Jak widać, czułośc (ang. *recall*) i precyzja  (ang. *precision*) dla klas o małej liczności nie są zbyt dobre. Spróbujemy je poprawić <b>równoważąc klasy w próbce uczącej.</b>
</div>

# Równoważenie klas

Poniżej zostaną zaprezentowane sposoby równoważenia klas, należące do kategorii opisywanych na wykładzie. Zastosujemy kilka z nich i sprawdzimy, czy dają oczekiwane rezultaty.

Zaczniemy od zaimportowania biblioteki, w której zawarte są odpowiednie narzędzia.

In [41]:
import imblearn

## Oversampling

Pierwszą z metod będzie 'dolosowywanie' obiektów z klasy mniejszościowej. Zrobimy to dwoma sposobami.

### Random

Pierwszy sposób dolosowtwania klasy mniejszościowej polega na losowym wyborze obiektów z klas mniejszościowych tak długo, aż poszczególne ilości się zrównoważą.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>RandomOverSampler</code>. </li>
        <li>Utwórz obiekt klasy <code>RandomOverSampler</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu RandomOverSamplera, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_ros, y_train_ros</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_ros: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [42]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=999)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

In [44]:
y_train_ros_classes = np.argmax(y_train_ros, axis=1)

# Calculate counts and proportions for a NumPy array
unique, counts = np.unique(y_train_ros_classes, return_counts=True)
proportions = counts / len(y_train_ros_classes)
ycnt_ros = dict(zip(unique, proportions))

Sprawdzenie poprawności wyniku:

In [45]:
for i in range(3):
    assert round(ycnt_ros[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m2_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [46]:
model2 = build_model()
model2.fit(X_train_ros, y_train_ros, epochs=5, batch_size=32, verbose=1)
m2_eval = model2.evaluate(X_test, y_test, return_dict=True)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


263/263 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4280 - loss: 1.0727
Epoch 2/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6512 - loss: 0.7867
Epoch 3/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7101 - loss: 0.6493
Epoch 4/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7581 - loss: 0.5807
Epoch 5/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7700 - loss: 0.5463
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7767 - loss: 0.5309


In [50]:
m2_eval

{'accuracy': 0.7820000052452087, 'loss': 0.529476523399353}

In [51]:
assert m2_eval['accuracy'] >= 0.7

In [47]:
y_pred = model2.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)
print(classification_report(y_test_classes, y_pred_classes))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

           0       0.93      0.80      0.86       694
           1       0.74      0.73      0.74       215
           2       0.37      0.77      0.50        91

    accuracy                           0.78      1000
   macro avg       0.68      0.77      0.70      1000
weighted avg       0.84      0.78      0.80      1000



<div class='alert alert-block alert-info'>
    Wyniki powinny się zmienić w stosunku do scenariusza bazowego - najprawdopodobniej spadła dokładność (ang. <i>accuracy</i>) ale wzrosła czułość i precyzja dla co najmniej jednej klasy mniejszościowej (1 i 2). To jest spodziewany efekt. Będziemy szukać dalej, czy inne procedury równoważenia zapewnią lepsze wyniki.
</div>

### SMOTE

Durgą procedurą równoważenia próbek, którą wykorzystamy będzie SMOTE, omawiane na wykładach. Ta metoda tworzy syntetyczne próbki, powtałe na przecięciu odcinków łączących obiekty z klasy mniejszościowej.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>SMOTE</code>. </li>
        <li>Utwórz obiekt klasy <code>SMOTE</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu SMOTE, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_smote, y_train_smote</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_smote: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [52]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=999)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [53]:
y_train_smote_classes = np.argmax(y_train_smote, axis=1)
unique, counts = np.unique(y_train_smote_classes, return_counts=True)
proportions = counts / len(y_train_smote_classes)
ycnt_smote = dict(zip(unique, proportions))

Sprawdzenie poprawności wyniku

In [54]:
for i in range(3):
    assert round(ycnt_smote[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m3_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [55]:
model3 = build_model() # utwórz nowy obiekt sieci neurnowej
model3.fit(X_train_smote, y_train_smote, epochs=5, batch_size=32, verbose=1) # Wyszkol przygotowany model
m3_eval = model3.evaluate(X_test, y_test, return_dict=True) # zapisz wartości metryk

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


263/263 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4728 - loss: 1.0323
Epoch 2/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6754 - loss: 0.7386
Epoch 3/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7327 - loss: 0.6081
Epoch 4/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7885 - loss: 0.5223
Epoch 5/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8110 - loss: 0.4598
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8108 - loss: 0.5066


In [59]:
m3_eval

{'accuracy': 0.8019999861717224, 'loss': 0.5070357322692871}

In [61]:
assert 0.7 <= m3_eval['accuracy']

In [64]:
y_pred = model3.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)
print(classification_report(y_test_classes, y_pred_classes)) # wyświetl raport klasyfikacji

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
              precision    recall  f1-score   support

           0       0.92      0.84      0.88       694
           1       0.79      0.68      0.73       215
           2       0.41      0.80      0.54        91

    accuracy                           0.80      1000
   macro avg       0.70      0.77      0.72      1000
weighted avg       0.84      0.80      0.81      1000



<div class='alert alert-block alert-info'>
    Wyniki powinny się zmienić w stosunku do scenariusza bazowego - najprawdopodobniej spadła dokładność (ang. <i>accuracy</i>) ale wzrosła czułość i precyzja dla co najmniej jednej klasy mniejszościowej (1 i 2). To jest spodziewany efekt. Będziemy szukać dalej, czy inne procedury równoważenia zapewnią lepsze wyniki.<br>
    Porównaj otrzymane wyniki z RandomOverSampling'iem. Czy jest lepiej, czy gorzej? Jeśli tak, to w jakim zakresie (w odniesieniu do której klasy?).
</div>

## Under sampling

Kolejna grupa procedur to zmniejszenie liczności klasy większościowej - odrzucenie nadmiarowych obserwaci, aby zredukować ją do takiej samej liczności, jak klasa mniejszościowa. Ta metoda niestety powoduje odrzucenie znacznej ilości użytecznych danych - z tego powodu może być czasem problematyczna.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>RandomUnderSampler</code>. </li>
        <li>Utwórz obiekt klasy <code>RandomUnderSampler</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu RandomUnderSampler, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_rus, y_train_rus</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_rus: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [62]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=999)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

In [63]:
y_train_rus_classes = np.argmax(y_train_rus, axis=1)
unique, counts = np.unique(y_train_rus_classes, return_counts=True)
proportions = counts / len(y_train_rus_classes)
ycnt_rus = dict(zip(unique, proportions))

Sprawdzenie poprawności wyniku:

In [ ]:
for i in range(3):
    assert round(ycnt_rus[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m4_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [ ]:
??? # wyszkol model
m4_eval = ??? # zapisz wratości metryk

In [ ]:
m4_eval

{'loss': 0.9615492820739746,
 'accuracy': 0.5580000281333923,
 'f1_score': 0.4704263508319855}

In [ ]:
assert 0.4 <= m4_eval['f1_score']
assert 0.4 <= m4_eval['accuracy']

In [ ]:
??? # wyświetl raport klasyfikacji

              precision    recall  f1-score   support

           0       0.85      0.77      0.81       694
           1       0.57      0.50      0.54       215
           2       0.26      0.53      0.35        91

    accuracy                           0.69      1000
   macro avg       0.56      0.60      0.57      1000
weighted avg       0.74      0.69      0.71      1000



<div class='alert alert-block alert-info'>
   W tym przypadku wyniki powinny być znacznie gorsze, niż przy wykorzystaniu wcześniejszych podejść oraz w modelu bazowym. Wynika to z faktu, że RandomUnderSampler odrzuca obiekty (rekordy), które mogą nieść ze sobą bardzo użyteczne informacje.
    <br>
    <br>
    Nie znaczy to, że UnderSampling nie jest przydatny - najcześciej wykorzystuje się go w sytuacjach, gdy mamy bardzo dużo danych, które niekoniecznie muszą być użyteczne (np. macierze rzadkie w systemach rekomendacyjnych, etc.).
    <br>
    <br>
    W tym konkretnym przypadku - raczej nam się nie przyda.
</div>

## SMOTETomek - upsampling i undersampling jedncześnie

Jak łatwo się domyślić, opisane wyżej metody można połączyć, stosując jednocześnie syntetyczny oversampling oraz redukcję niektórych obserwacji z klasy większościowej. Spróbujmy i sprawdźmy, czy ta metoda da lepsze rezultaty niż np. wyłącznie SMOTE.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>SMOTETomek</code>. </li>
        <li>Utwórz obiekt klasy <code>SMOTETomek</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu SMOTETomek, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_smotet, y_train_smotet</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_smotet: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [ ]:
smotet = ???
X_train_smotet, y_train_smotet = ???

C:\Users\fwojcik\Anaconda3\envs\tf\lib\site-packages\sklearn\utils\validation.py:70: FutureWarning: Pass classes=[0 1 2] as keyword args. From version 1.0 (renaming of 0.25) passing these as positional arguments will result in an error
  warnings.warn(f"Pass {args_msg} as keyword args. From version "


In [ ]:
ycnt_smotet = ???

Sprawdzenie poprawności wyniku:

In [ ]:
for i in range(3):
    assert round(ycnt_smotet[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m5_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [ ]:
??? # wyszkol model
m5_eval = ??? # zapisz wartości metryk

Epoch 1/5
262/262 [==============================] - 0s 2ms/step - loss: 1.0188 - accuracy: 0.4921 - f1_score: 0.4855
Epoch 2/5
262/262 [==============================] - 0s 2ms/step - loss: 0.6948 - accuracy: 0.7076 - f1_score: 0.7077
Epoch 3/5
262/262 [==============================] - 0s 1ms/step - loss: 0.5461 - accuracy: 0.7676 - f1_score: 0.7677
Epoch 4/5
262/262 [==============================] - 0s 1ms/step - loss: 0.4759 - accuracy: 0.7993 - f1_score: 0.7991
Epoch 5/5
32/32 [==============================] - 0s 1ms/step - loss: 0.4988 - accuracy: 0.7930 - f1_score: 0.7129


In [ ]:
m5_eval

{'loss': 0.498818039894104,
 'accuracy': 0.7929999828338623,
 'f1_score': 0.7128863334655762}

In [ ]:
assert 0.7 <= m5_eval['accuracy']
assert 0.65 <= m5_eval['f1_score']

In [ ]:
??? # wyświel raport klasyfikacji

              precision    recall  f1-score   support

           0       0.93      0.81      0.87       694
           1       0.77      0.72      0.74       215
           2       0.39      0.82      0.53        91

    accuracy                           0.79      1000
   macro avg       0.70      0.78      0.71      1000
weighted avg       0.84      0.79      0.81      1000



<div class='alert alert-block alert-info'>
 W zależności od przebiegu uczenia, wyniki będą zbliżone lub nieznacznie odbiegające od tych, któe daje SMOTE. Znacząco powinna wzrosnąć czułość (wykrywalność) klas mniejszościowych w stosunku do scenariusza bazowego, kosztem precyzji. Innymi słowy - model częściej znajduje obiekty klasy mniejszościowej, ale jednocześnie zaczyna się mylić robiąć takie przypisania.
</div>

# Nadawanie wag klasom

Jeszcze jednym sposobem na szkolenie sieci neuronowej do rozpoznawania obiektów klasy mniejszościowej, jest nadanie wag poszczególnym klasom. Działa to w sposób następujący:

1. Każdej klasie nadajemy jakąś wagę.
2. Podczas procesu uczenia się, dla obiektów danej klasy, funkcja kosztu (np. entropia krzyżowa, ang. *Cross entropy*) jest wymnażana przez tą wagę
3. W ten sposób, obiekty określonej klasy mogą ważyć więcej lub mniej w przypadku ich błędnej klasyfikacji i tym samym silniej lub słabiej wpływać na dopasowanie funkcji kosztu.


W naszym przykładzie spróbujemy **bez równoważenia zbioru** nadać klasie o najmniejszej liczności (2) wagę = 0.5, drugiej mniej licznej klasie (1) wagę 0.35 i klasie więszkościowej wagę 0.15, aby położyć większy nacisk na 1 i 2.

<div class='alert alert-block alert-warning'>
    <b>Zadanie:</b> utwórz słównik, określający wagi poszczgólnych klas w sposób następujący:
    <il>
        <li>Klasa 0: waga 0.15</li>
        <li>Klasa 1: waga 0.35</li>
        <li>Klasa 2: waga 0.5</li>
    </il>
</div>

In [ ]:
class_weights = ???

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na <b>PODSTAWOWYM zbiorze treningowym</b> bez równoważenia </li>
        <li>Do funkcji <code>fit()</code> sieci neuronowej, przekaż dodatkowy argument <code>class_weights=</code>zawierający określone wcześniej wagi klas.<li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m6_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [ ]:
??? # wyszkol model
m6_eval = ??? # zapisz wratości metryk

Epoch 1/5
125/125 [==============================] - 0s 2ms/step - loss: 0.2373 - accuracy: 0.6480 - f1_score: 0.3922
Epoch 2/5
125/125 [==============================] - 0s 2ms/step - loss: 0.2084 - accuracy: 0.7287 - f1_score: 0.5407
Epoch 3/5
125/125 [==============================] - 0s 2ms/step - loss: 0.1868 - accuracy: 0.7473 - f1_score: 0.5958
Epoch 4/5
125/125 [==============================] - 0s 2ms/step - loss: 0.1680 - accuracy: 0.7717 - f1_score: 0.6413
Epoch 5/5
32/32 [==============================] - 0s 1ms/step - loss: 0.5925 - accuracy: 0.7650 - f1_score: 0.6438


In [ ]:
m6_eval

{'loss': 0.5925179123878479,
 'accuracy': 0.7649999856948853,
 'f1_score': 0.6438450217247009}

Sprawdzenie poprawności wyniku

In [ ]:
assert 0.65 <= m6_eval['accuracy']
assert 0.6 <= m6_eval['f1_score']

In [ ]:
??? # wyświetl raport klasyfikacji

              precision    recall  f1-score   support

           0       0.88      0.77      0.82       694
           1       0.86      0.40      0.55       215
           2       0.24      0.76      0.36        91

    accuracy                           0.69      1000
   macro avg       0.66      0.64      0.58      1000
weighted avg       0.82      0.69      0.72      1000



<div class='alert alert-block alert-info'>
 Otrzymane wyniki nie wyglądają na istotnie lepsze/gorsze od tych, otrzymywanych podczas równoważenia zbioru. Zastosowanie wag dla klas jest po prostu kolejnym narzędziem, po które warto sięgnąć w sytuacji, gdy mamy do czynienia z niezbalansowanymi klasami w zbiorze uczącym.
</div>

# Dalsze eksperymenty

Jeśli chcesz, przepowadź dalesze eksperymenty na przedstawionym zbiorze danych, obejmujące np. poszukiwanie odpowiedniej architektury sieci oraz hiperparametrów. Spróbuj zastosować różne metody równoważenia zbiorów, z odmiennymi parametrami. Może uda Ci się uzyskać zadowalajace rezultaty?

# Task
Fix the `SyntaxError` in the `build_model` function by changing the `metrics` list from `['accuracy', average='macro')]` to `['accuracy']` because `tensorflow_addons` (which contains `F1Score`) is not available in the current environment.

## Modify build_model function

### Subtask:
Fix the `SyntaxError` in the `metrics` list within the `build_model` function by setting it to `['accuracy']`, as `tensorflow_addons` is not available.


## Summary:

There was no solving process provided to summarize. Please provide the solving process to generate the summary.
